# 01_prepare_chunks_by_sheets

Первый этап пайплайна.

Этот ноутбук делает подготовку данных:

```text
Excel → requirements_from_excel.md → chunks.json
```

Главное правило этой версии:

```text
один лист Excel = один чанк
```

То есть:

```text
Лист "поставка, купля-прод." → chunk 1
Лист "подряд"                → chunk 2
Лист "ГПД с ФЛ"              → chunk 3
...
```

LLM здесь не используется.

## 1. Установка библиотек

In [ ]:
%pip install pandas openpyxl

## 2. Импорты и настройки

In [ ]:
from pathlib import Path
import json
import re

import pandas as pd


INPUT_XLSX = Path("Исходные_требования (3).xlsx")
MARKDOWN_OUTPUT = Path("requirements_from_excel.md")
CHUNKS_OUTPUT = Path("chunks.json")

SKIP_SHEETS = {"Пример заполнения модели данных"}

print("INPUT_XLSX:", INPUT_XLSX)
print("MARKDOWN_OUTPUT:", MARKDOWN_OUTPUT)
print("CHUNKS_OUTPUT:", CHUNKS_OUTPUT)
print("CHUNKING_STRATEGY: one_sheet_one_chunk")

## 3. Проверка, что Excel-файл найден

In [ ]:
if not INPUT_XLSX.exists():
    raise FileNotFoundError(
        f"Файл не найден: {INPUT_XLSX}. "
        "Положи Excel-файл в одну папку с ноутбуком или измени INPUT_XLSX."
    )

print("Excel-файл найден.")

## 4. Функции очистки ячеек

In [ ]:
def clean_cell(value) -> str:
    """Очищает значение ячейки Excel для Markdown."""

    if pd.isna(value):
        return ""

    text = str(value)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()


def escape_markdown_cell(value) -> str:
    """Экранирует ячейку, чтобы Markdown-таблица не ломалась."""

    text = clean_cell(value)
    text = text.replace("|", "\\|")
    text = text.replace("\n", "<br>")
    return text

## 5. Чтение Excel и сохранение полного Markdown

In [ ]:
def dataframe_to_markdown(sheet_name: str, df: pd.DataFrame) -> str:
    """Преобразует один лист Excel в Markdown-таблицу."""

    markdown_parts = []
    markdown_parts.append(f"# Лист: {sheet_name}")
    markdown_parts.append(f"Количество строк листа: {len(df)}")
    markdown_parts.append("")

    if df.empty:
        markdown_parts.append("_Пустой лист_")
        return "\n".join(markdown_parts)

    headers = [
        escape_markdown_cell(col) or f"Колонка {i + 1}"
        for i, col in enumerate(df.columns)
    ]

    markdown_parts.append("| " + " | ".join(headers) + " |")
    markdown_parts.append("| " + " | ".join(["---"] * len(headers)) + " |")

    for row_index, row in df.iterrows():
        values = [escape_markdown_cell(row[col]) for col in df.columns]
        markdown_parts.append("| " + " | ".join(values) + " |")

    return "\n".join(markdown_parts)


def excel_to_markdown(input_xlsx: Path, output_md: Path) -> dict:
    """
    Читает все листы Excel и сохраняет их в Markdown.

    Возвращает словарь:
    {
      "имя листа": DataFrame
    }
    """

    xls = pd.ExcelFile(input_xlsx)
    sheets = {}
    markdown_parts = []

    for sheet_name in xls.sheet_names:
        if sheet_name in SKIP_SHEETS:
            print(f"Пропускаю лист-пример: {sheet_name}")
            continue

        df = pd.read_excel(input_xlsx, sheet_name=sheet_name, dtype=str)
        df = df.dropna(how="all")
        df = df.dropna(axis=1, how="all")
        df = df.fillna("")

        sheets[sheet_name] = df
        markdown_parts.append(dataframe_to_markdown(sheet_name, df))
        markdown_parts.append("")

    output_md.write_text("\n".join(markdown_parts), encoding="utf-8")
    return sheets


sheets = excel_to_markdown(INPUT_XLSX, MARKDOWN_OUTPUT)

print(f"Markdown сохранён: {MARKDOWN_OUTPUT}")
print(f"Количество листов для обработки: {len(sheets)}")

for sheet_name, df in sheets.items():
    print(f"- {sheet_name}: {len(df)} строк, {len(df.columns)} колонок")

## 6. Создание чанков по листам

In [ ]:
def create_chunks_by_sheets(sheets: dict) -> list[dict]:
    """
    Делит документ на чанки по листам Excel.

    Один чанк = один лист Excel.
    """

    chunks = []
    chunk_id = 1

    for sheet_name, df in sheets.items():
        chunk_text = dataframe_to_markdown(sheet_name, df)

        chunks.append(
            {
                "chunk_id": chunk_id,
                "source_section": sheet_name,
                "sheet_name": sheet_name,
                "rows_count": len(df),
                "chunking_strategy": "one_sheet_one_chunk",
                "text": chunk_text
            }
        )

        chunk_id += 1

    return chunks

## 7. Сохранение `chunks.json`

In [ ]:
chunks = create_chunks_by_sheets(sheets)

CHUNKS_OUTPUT.write_text(
    json.dumps(chunks, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print(f"Файл чанков сохранён: {CHUNKS_OUTPUT}")
print(f"Количество чанков: {len(chunks)}")

for chunk in chunks:
    print(
        f"chunk={chunk['chunk_id']:03d} | "
        f"sheet={chunk['sheet_name']} | "
        f"rows_count={chunk['rows_count']} | "
        f"chars={len(chunk['text'])}"
    )

## 8. Быстрый просмотр первого чанка

In [ ]:
print(chunks[0]["text"][:3000])